In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Get the current working directory of the notebook
current_notebook_dir = os.getcwd()

# Get the parent directory
parent_dir = os.path.abspath(os.path.join(current_notebook_dir, os.pardir))

# Add the parent directory to the Python path
sys.path.append(parent_dir)

In [7]:
from matplotlib import pyplot as plt
import random
import math
import re
import pandas as pd
from PIL import Image
import numpy as np
import cv2

from torchvision.transforms import v2 as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch
from torchvision.transforms import ToPILImage
import torchvision
from sklearn.model_selection import train_test_split

In [4]:
image_metadata_path = "../Deepfake-Eval-2024/image-metadata-publish.csv"
image_dir_path = '../Deepfake-Eval-2024/image-data'

In [5]:
image_df = pd.read_csv(image_metadata_path)

In [6]:
image_df.head()

,Filename,Date,Ground Truth,Public Comments,Finetuning Set
0,9s4baJE29mP6-0Nk-KJRoS7TV34.jpg,########,Fake,some faces are distorted.,Train
1,PIL0v47Sw1bH3dI7Ob6k7GEFcPk.webp,########,Fake,"Image has fake feel of maniapulation, almost c...",Test
2,5s8g_7LCvNvOzTiBk549-ZcAw0I.webp,########,Fake,"Image has an almost cartoonish feel, digitally...",Train
3,7r-hud89IPwGjWz2MWoQovu9J4w.webp,########,Fake,"Image has an almost cartoonish feel, digitally...",Train
4,tvU3wuaKHstYLvwxsKmPyhqBcd0.jpg,########,Fake,"Obvious because overly smooth texture, artific...",Train


In [9]:
# Min and max resolution for deepfake images
min_resolution = float('inf')
min_height = float('inf')
min_width = float('inf')
max_resolution = float('-inf')
max_height = float('-inf')
max_width = float('-inf')

for idx, row in image_df.iterrows():
    image_path = os.path.join(image_dir_path, row['Filename'])
    # read image in grayscale
    img = cv2.imread(image_path,0)
    height, width = img.shape[:2]
    resolution = height * width
    if resolution < min_resolution:
        min_resolution = resolution
        min_height = height
        min_width = width
    if resolution > max_resolution:
        max_resolution = resolution
        max_height = height
        max_width = width

print(f'Min resolution: {min_resolution}, {min_width} x {min_height}')
print(f'Max resolution: {max_resolution}, {max_width} x {max_height}')

Premature end of JPEG file


Min resolution: 26600, 200 x 133
Max resolution: 44761088, 5464 x 8192


In [12]:
new_dir = '../Deepfake-Eval-2024/image-data-rescaled'
os.makedirs(new_dir, exist_ok=True)

In [15]:
# Go though the entire list of images
for index, row in image_df.iterrows():
    if row['Filename'] in ['c1DVk98VHjpJOPZFOPS9NgCBrRY.jpg', 't1ChhPlw2FBnI3V4AFk9vhYn1qg.jpeg']:
        continue
    save_path = os.path.join(new_dir, row['Filename'])

    # Print progress
    if (index + 1) % 100 == 0:
        print(f'Processed {index + 1} images')

    # Load image and bbox
    img = Image.open(os.path.join(image_dir_path, row['Filename']))
    if img.mode != 'RGB': # Convert to RGB if it is not in RGB
        img = img.convert('RGB')

    # Resize to 384 pixels on the small side
    rescaled_img = transforms.functional.resize(img, 384)
    rescaled_img.save(save_path)

Processed 100 images
Processed 200 images
Processed 300 images
Processed 400 images
Processed 500 images
Processed 600 images
Processed 700 images
Processed 800 images
Processed 900 images
Processed 1000 images
Processed 1100 images
Processed 1200 images
Processed 1300 images
Processed 1400 images
Processed 1500 images
Processed 1600 images
Processed 1700 images
Processed 1800 images
Processed 1900 images


In [17]:
# Min and max resolution for deepfake images
min_resolution = float('inf')
min_height = float('inf')
min_width = float('inf')
max_resolution = float('-inf')
max_height = float('-inf')
max_width = float('-inf')

for idx, row in image_df.iterrows():
    if row['Filename'] in ['c1DVk98VHjpJOPZFOPS9NgCBrRY.jpg', 't1ChhPlw2FBnI3V4AFk9vhYn1qg.jpeg']:
        continue
    image_path = os.path.join(new_dir, row['Filename'])
    # read image in grayscale
    img = cv2.imread(image_path,0)
    height, width = img.shape[:2]
    resolution = height * width
    if resolution < min_resolution:
        min_resolution = resolution
        min_height = height
        min_width = width
    if resolution > max_resolution:
        max_resolution = resolution
        max_height = height
        max_width = width

print(f'Min resolution: {min_resolution}, {min_width} x {min_height}')
print(f'Max resolution: {max_resolution}, {max_width} x {max_height}')

Min resolution: 147456, 384 x 384
Max resolution: 461568, 1202 x 384
